# Day 2 -- Analytical SQL

Each lab question in its own cell, run through SQLAlchemy into a pandas
DataFrame.

The SQL here is the same text as `queries/day02_analytics.sql`. That file
is the artifact you would hand to another engineer; this notebook is for
running and eyeballing results.

Keyword explanations: `queries/README.md`

In [21]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5433')}"
    f"/{os.getenv('POSTGRES_DB')}"
)

engine = create_engine(URL)

# Fail now with a clear message rather than inside the first query.
with engine.connect() as conn:
    print(conn.execute(text("select version()")).scalar())

PostgreSQL 17.11 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit


In [22]:
def q(sql: str) -> pd.DataFrame:
    """Run SQL, return a DataFrame."""
    return pd.read_sql(text(sql), engine)


pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

## Q1. Monthly revenue and order count, 2017-2018

`count(distinct o.order_id)` is load-bearing: the join to `order_items`
repeats each order once per item, so a plain `count(*)` overstates orders
by about 14%.

In [23]:
q1 = q("""
select
    date_trunc('month', o.order_purchase_timestamp)::date as month,
    sum(oi.price)                                         as revenue,
    count(distinct o.order_id)                            as orders
from source.orders o
join source.order_items oi on oi.order_id = o.order_id
where o.order_purchase_timestamp >= '2017-01-01'
  and o.order_purchase_timestamp <  '2019-01-01'
  and o.order_status <> 'canceled'
group by 1
order by 1
""")
q1.head(10)

,month,revenue,orders
0,2017-01-01,120098.27,787
1,2017-02-01,244959.35,1718
2,2017-03-01,368341.32,2617
3,2017-04-01,353842.98,2377
4,2017-05-01,503159.19,3640
5,2017-06-01,429916.61,3205
6,2017-07-01,492287.30,3946
7,2017-08-01,568245.79,4272
8,2017-09-01,621415.91,4227
9,2017-10-01,660179.62,4547


In [24]:
print(f"months  : {len(q1)}")
print(f"revenue : {q1.revenue.sum():,.2f}")
print(f"orders  : {q1.orders.sum():,}")

months  : 21
revenue : 13,449,674.68
orders  : 97,906


## Q2. Month-over-month revenue growth %

`lag(revenue)` is computed in its own CTE so the final select can name it
instead of repeating the window expression three times.

Watch September 2018: the dataset stops, so growth reads -99.98%. That is
a collection artifact, not a business event.

In [25]:
q2 = q("""
with monthly as (
    select
        date_trunc('month', o.order_purchase_timestamp)::date as month,
        sum(oi.price)                                         as revenue,
        count(distinct o.order_id)                            as orders
    from source.orders o
    join source.order_items oi on oi.order_id = o.order_id
    where o.order_purchase_timestamp >= '2017-01-01'
      and o.order_purchase_timestamp <  '2019-01-01'
      and o.order_status <> 'canceled'
    group by 1
),
with_prev as (
    select
        month,
        revenue,
        orders,
        lag(revenue) over (order by month) as prev_revenue
    from monthly
)
select
    month,
    revenue,
    prev_revenue,
    round(100.0 * (revenue - prev_revenue) / nullif(prev_revenue, 0), 2) as growth_pct
from with_prev
order by month
""")
q2.head(10)

,month,revenue,prev_revenue,growth_pct
0,2017-01-01,120098.27,NaN,NaN
1,2017-02-01,244959.35,120098.27,103.97
2,2017-03-01,368341.32,244959.35,50.37
3,2017-04-01,353842.98,368341.32,-3.94
4,2017-05-01,503159.19,353842.98,42.20
5,2017-06-01,429916.61,503159.19,-14.56
6,2017-07-01,492287.30,429916.61,14.51
7,2017-08-01,568245.79,492287.30,15.43
8,2017-09-01,621415.91,568245.79,9.36
9,2017-10-01,660179.62,621415.91,6.24


## Q3. Top 3 categories by revenue in each state

`rank() over (partition by state order by revenue desc)` restarts the
numbering for every state.

The ranking is computed in one CTE and filtered in the next, because
`where` runs before window functions exist.

In [26]:
q3 = q("""
with category_state_revenue as (
    select
        c.customer_state as state,
        coalesce(t.product_category_name_english,
                 p.product_category_name,
                 '(uncategorised)') as category,
        sum(oi.price)              as revenue,
        count(distinct o.order_id) as orders
    from source.orders o
    join source.order_items oi on oi.order_id   = o.order_id
    join source.customers   c  on c.customer_id = o.customer_id
    join source.products    p  on p.product_id  = oi.product_id
    left join source.product_category_translation t
           on t.product_category_name = p.product_category_name
    where o.order_purchase_timestamp >= '2017-01-01'
      and o.order_purchase_timestamp <  '2019-01-01'
      and o.order_status <> 'canceled'
    group by 1, 2
),
ranked as (
    select
        state,
        category,
        revenue,
        orders,
        rank() over (partition by state order by revenue desc) as rnk
    from category_state_revenue
)
select state, rnk, category, revenue, orders
from ranked
where rnk <= 3
order by state, rnk
""")
q3.head(10)

,state,rnk,category,revenue,orders
0,AC,1,sports_leisure,1677.46,9
1,AC,2,watches_gifts,1389.60,4
2,AC,3,health_beauty,1386.58,6
3,AL,1,health_beauty,12780.26,63
4,AL,2,watches_gifts,11683.27,35
5,AL,3,computers_accessories,7681.86,33
6,AM,1,health_beauty,2776.03,15
7,AM,2,garden_tools,1980.88,5
8,AM,3,computers_accessories,1960.74,11
9,AP,1,computers_accessories,2149.52,7


In [27]:
print(f"rows   : {len(q3)}")
print(f"states : {q3.state.nunique()}")
print("\nmost common category at rank 1:")
print(q3[q3.rnk == 1].category.value_counts().head())

rows   : 81
states : 27

most common category at rank 1:
category
health_beauty            14
watches_gifts             6
sports_leisure            3
bed_bath_table            2
computers_accessories     1
Name: count, dtype: int64


## Q4. Running total of revenue per seller

`sum()` becomes a window function by adding `over`. The frame clause
`rows between unbounded preceding and current row` makes it cumulative.

Limited to the top 5 sellers so the output stays readable.

In [28]:
q4 = q("""
with seller_monthly as (
    select
        oi.seller_id,
        date_trunc('month', o.order_purchase_timestamp)::date as month,
        sum(oi.price) as revenue
    from source.orders o
    join source.order_items oi on oi.order_id = o.order_id
    where o.order_status <> 'canceled'
    group by 1, 2
),
top_sellers as (
    select seller_id
    from seller_monthly
    group by 1
    order by sum(revenue) desc
    limit 5
)
select
    sm.seller_id,
    sm.month,
    sm.revenue,
    sum(sm.revenue) over (
        partition by sm.seller_id
        order by sm.month
        rows between unbounded preceding and current row
    ) as running_total
from seller_monthly sm
join top_sellers ts on ts.seller_id = sm.seller_id
order by sm.seller_id, sm.month
""")
q4.head(10)

,seller_id,month,revenue,running_total
0,4869f7a5dfa277a7dca6462dcf3b52b2,2017-03-01,1385.00,1385.00
1,4869f7a5dfa277a7dca6462dcf3b52b2,2017-05-01,374.00,1759.00
2,4869f7a5dfa277a7dca6462dcf3b52b2,2017-06-01,5064.85,6823.85
3,4869f7a5dfa277a7dca6462dcf3b52b2,2017-07-01,7201.30,14025.15
4,4869f7a5dfa277a7dca6462dcf3b52b2,2017-08-01,11386.78,25411.93
5,4869f7a5dfa277a7dca6462dcf3b52b2,2017-09-01,12536.40,37948.33
6,4869f7a5dfa277a7dca6462dcf3b52b2,2017-10-01,16548.70,54497.03
7,4869f7a5dfa277a7dca6462dcf3b52b2,2017-11-01,18698.20,73195.23
8,4869f7a5dfa277a7dca6462dcf3b52b2,2017-12-01,17627.80,90823.03
9,4869f7a5dfa277a7dca6462dcf3b52b2,2018-01-01,19954.20,110777.23


## Q5. Actual vs estimated delivery time, by state

`date - date` gives integer days. `filter (where ...)` applies a condition
to a single aggregate.

In [29]:
q5 = q("""
select
    c.customer_state as state,
    count(*)         as delivered_orders,
    round(avg(o.order_delivered_customer_date::date
              - o.order_purchase_timestamp::date), 1) as avg_actual_days,
    round(avg(o.order_estimated_delivery_date::date
              - o.order_purchase_timestamp::date), 1) as avg_promised_days,
    round(avg(o.order_delivered_customer_date::date
              - o.order_estimated_delivery_date::date), 1) as avg_days_vs_promise,
    round(100.0 * count(*) filter (
              where o.order_delivered_customer_date
                  > o.order_estimated_delivery_date
          ) / count(*), 2) as late_pct
from source.orders o
join source.customers c on c.customer_id = o.customer_id
where o.order_status = 'delivered'
  and o.order_delivered_customer_date is not null
group by 1
order by avg_actual_days desc
""")
q5.head(10)

,state,delivered_orders,avg_actual_days,avg_promised_days,avg_days_vs_promise,late_pct
0,RR,41,29.3,46.6,-17.3,12.20
1,AP,67,27.2,46.9,-19.7,4.48
2,AM,145,26.4,45.9,-19.6,4.14
3,AL,397,24.5,33.2,-8.7,23.93
4,PA,946,23.7,37.8,-14.1,12.37
5,SE,335,21.5,31.5,-10.0,15.22
6,MA,717,21.5,31.1,-9.6,19.67
7,CE,1279,21.2,32.0,-10.8,15.32
8,AC,80,21.0,41.7,-20.7,3.75
9,PB,517,20.4,33.6,-13.3,11.03


Every state beats its promise on average, yet late rates reach 24%.
The mean hides the tail: most orders arrive very early, a minority very
late. Always pair an average with a rate or a percentile.

## Q6. Repeat customer rate

Must group by `customer_unique_id`. `customer_id` is generated per order,
so grouping by it reports a 0% repeat rate with no error.

In [30]:
q6 = q("""
with customer_orders as (
    select
        c.customer_unique_id,
        count(distinct o.order_id) as orders
    from source.orders o
    join source.customers c on c.customer_id = o.customer_id
    where o.order_status <> 'canceled'
    group by 1
)
select
    count(*)                                  as total_customers,
    count(*) filter (where orders > 1)        as repeat_customers,
    round(100.0 * count(*) filter (where orders > 1) / count(*), 2) as repeat_rate_pct,
    max(orders)                               as most_orders_by_one_customer,
    round(avg(orders), 3)                     as avg_orders_per_customer
from customer_orders
""")
q6.head(10)

,total_customers,repeat_customers,repeat_rate_pct,most_orders_by_one_customer,avg_orders_per_customer
0,95560,2924,3.06,17,1.034


In [31]:
q6_wrong = q("""
with wrong as (
    select o.customer_id, count(distinct o.order_id) as orders
    from source.orders o
    where o.order_status <> 'canceled'
    group by 1
)
select
    count(*)                           as total_customers,
    count(*) filter (where orders > 1) as repeat_customers,
    round(100.0 * count(*) filter (where orders > 1) / count(*), 2) as repeat_rate_pct
from wrong
""")
q6_wrong.head(10)

,total_customers,repeat_customers,repeat_rate_pct
0,98816,0,0.0


## Q7. Cohort retention

Assign each customer to the month of their first order, then count how
many ordered again 1, 2 and 3 months later.

`filter (where month_offset = n)` is the SQL pivot idiom: one aggregate
per output column.

In [32]:
q7 = q("""
with customer_first_order as (
    select
        c.customer_unique_id,
        date_trunc('month', min(o.order_purchase_timestamp))::date as cohort_month
    from source.orders o
    join source.customers c on c.customer_id = o.customer_id
    where o.order_status <> 'canceled'
    group by 1
),
orders_with_offset as (
    select
        f.customer_unique_id,
        f.cohort_month,
        (extract(year  from date_trunc('month', o.order_purchase_timestamp))
         - extract(year  from f.cohort_month)) * 12
      + (extract(month from date_trunc('month', o.order_purchase_timestamp))
         - extract(month from f.cohort_month)) as month_offset
    from source.orders o
    join source.customers c     on c.customer_id        = o.customer_id
    join customer_first_order f on f.customer_unique_id = c.customer_unique_id
    where o.order_status <> 'canceled'
)
select
    cohort_month,
    count(distinct customer_unique_id) filter (where month_offset = 0) as cohort_size,
    count(distinct customer_unique_id) filter (where month_offset = 1) as month_1,
    count(distinct customer_unique_id) filter (where month_offset = 2) as month_2,
    count(distinct customer_unique_id) filter (where month_offset = 3) as month_3,
    round(100.0 * count(distinct customer_unique_id) filter (where month_offset = 1)
          / nullif(count(distinct customer_unique_id) filter (where month_offset = 0), 0), 2) as m1_pct
from orders_with_offset
group by 1
order by 1
""")
q7.head(10)

,cohort_month,cohort_size,month_1,month_2,month_3,m1_pct
0,2016-09-01,2,0,0,0,0.00
1,2016-10-01,297,0,0,0,0.00
2,2016-12-01,1,1,0,0,100.00
3,2017-01-01,762,3,2,1,0.39
4,2017-02-01,1735,4,5,2,0.23
5,2017-03-01,2603,13,9,10,0.50
6,2017-04-01,2334,14,5,4,0.60
7,2017-05-01,3571,17,18,14,0.48
8,2017-06-01,3126,14,11,13,0.45
9,2017-07-01,3868,20,13,10,0.52


Month-1 retention is under 1%. Normal for a marketplace selling furniture
and watches -- people do not rebuy a bed frame monthly. Note 2016-11 is
missing entirely: a gap in the dataset, not a month without sales.

## Q8. Orders with no payment row (anti-join)

`not exists` is the safe form. `not in` returns zero rows if the subquery
contains a single NULL.

In [33]:
q8 = q("""
select
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    (select count(*) from source.order_items i where i.order_id = o.order_id) as items
from source.orders o
where not exists (
    select 1
    from source.order_payments p
    where p.order_id = o.order_id
)
order by o.order_purchase_timestamp
""")
q8

,order_id,order_status,order_purchase_timestamp,items
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,3


## Q9. Deduplication

Runs on a scratch copy in `staging`, never on `source`. `ctid` is the
physical row id -- the only thing distinguishing two identical rows.

In [34]:
from sqlalchemy import text as _text  # noqa: E402

with engine.begin() as conn:
    conn.execute(_text("drop table if exists staging.sellers_dedup_demo"))
    conn.execute(_text(
        "create table staging.sellers_dedup_demo as select * from source.sellers"))
    conn.execute(_text(
        "insert into staging.sellers_dedup_demo "
        "select * from source.sellers order by seller_id limit 100"))

before = q("""
select count(*) as total_rows,
       count(distinct seller_id) as distinct_sellers,
       count(*) - count(distinct seller_id) as duplicate_rows
from staging.sellers_dedup_demo
""")
print("before dedup:")
before

before dedup:


,total_rows,distinct_sellers,duplicate_rows
0,3195,3095,100


In [35]:
with engine.begin() as conn:
    conn.execute(_text("""
        delete from staging.sellers_dedup_demo a
        using (
            select ctid,
                   row_number() over (partition by seller_id order by ctid) as rn
            from staging.sellers_dedup_demo
        ) dup
        where a.ctid = dup.ctid
          and dup.rn > 1
    """))

after = q("""
select count(*) as total_rows,
       count(distinct seller_id) as distinct_sellers,
       count(*) - count(distinct seller_id) as duplicate_rows
from staging.sellers_dedup_demo
""")
print("after dedup:")
after

after dedup:


,total_rows,distinct_sellers,duplicate_rows
0,3095,3095,0


## Q10. EXPLAIN ANALYZE and an index

`customer_city` has no index, so the filter forces a sequential scan.
Read the plan inside out -- the most indented node runs first.

In [36]:
PLAN_SQL = """
select count(*), sum(oi.price)
from source.orders o
join source.customers   c  on c.customer_id = o.customer_id
join source.order_items oi on oi.order_id   = o.order_id
where c.customer_city = 'sao paulo'
  and o.order_status <> 'canceled'
"""

# Drop first, so re-running the notebook measures a real before/after
# rather than two post-index runs.
with engine.begin() as conn:
    conn.execute(_text("drop index if exists source.idx_customers_city"))
    conn.execute(_text("analyze source.customers"))

with engine.connect() as conn:
    rows = conn.execute(_text(f"explain (analyze, buffers) {PLAN_SQL}")).fetchall()
print("BEFORE INDEX")
print("\n".join(r[0] for r in rows))

BEFORE INDEX
Finalize Aggregate  (cost=9527.10..9527.11 rows=1 width=40) (actual time=30.489..33.345 rows=1 loops=1)
  Buffers: shared hit=5719
  ->  Gather  (cost=9526.98..9527.09 rows=1 width=40) (actual time=30.200..33.332 rows=2 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=5719
        ->  Partial Aggregate  (cost=8526.98..8526.99 rows=1 width=40) (actual time=28.057..28.061 rows=1 loops=2)
              Buffers: shared hit=5719
              ->  Parallel Hash Join  (cost=5182.18..8474.75 rows=10444 width=6) (actual time=17.175..27.341 rows=8836 loops=2)
                    Hash Cond: (oi.order_id = o.order_id)
                    Buffers: shared hit=5719
                    ->  Parallel Seq Scan on order_items oi  (cost=0.00..2982.65 rows=66265 width=39) (actual time=0.008..3.031 rows=56325 loops=2)
                          Buffers: shared hit=2320
                    ->  Parallel Hash  (cost=5066.94..5066.94 rows=9219 width=33) (act

In [37]:
with engine.begin() as conn:
    conn.execute(_text(
        "create index if not exists idx_customers_city "
        "on source.customers (customer_city)"))
    conn.execute(_text("analyze source.customers"))

with engine.connect() as conn:
    rows = conn.execute(_text(f"explain (analyze, buffers) {PLAN_SQL}")).fetchall()
print("AFTER INDEX")
print("\n".join(r[0] for r in rows))

AFTER INDEX
Finalize Aggregate  (cost=9091.37..9091.38 rows=1 width=40) (actual time=37.424..41.192 rows=1 loops=1)
  Buffers: shared hit=5644 read=15
  ->  Gather  (cost=9091.25..9091.36 rows=1 width=40) (actual time=36.897..41.173 rows=2 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=5644 read=15
        ->  Partial Aggregate  (cost=8091.25..8091.26 rows=1 width=40) (actual time=33.957..33.965 rows=1 loops=2)
              Buffers: shared hit=5644 read=15
              ->  Parallel Hash Join  (cost=4747.82..8039.66 rows=10319 width=6) (actual time=21.730..33.194 rows=8836 loops=2)
                    Hash Cond: (oi.order_id = o.order_id)
                    Buffers: shared hit=5644 read=15
                    ->  Parallel Seq Scan on order_items oi  (cost=0.00..2982.65 rows=66265 width=39) (actual time=0.008..3.354 rows=56325 loops=2)
                          Buffers: shared hit=2320
                    ->  Parallel Hash  (cost=4633.95..4

`Parallel Seq Scan` on customers (41,950 rows discarded) becomes a
`Bitmap Index Scan`, and runtime drops from ~46 ms to ~36 ms.

Only 21% faster, because `orders` and `order_items` are still scanned
sequentially and dominate the cost. Read the plan before adding an index,
or you optimise something that was not the bottleneck.

In [38]:
engine.dispose()